<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/30_num_int/10_first_order.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)



In [ ]:
# 그래프, 수학 기능 추가
# Add graph and math features
import matplotlib.pyplot as plt
import numpy as np



# 1차 적분<br>First Order Numerical Integration



[![Trapezoidal sums | Accumulation and Riemann sums | AP Calculus AB | Khan Academy](https://i.ytimg.com/vi/1p0NHR5w0Lc/hqdefault.jpg)](https://www.youtube.com/watch?v=1p0NHR5w0Lc)



이 장에서는 $\int_0^1 e^x \, dx$를 주요 예제로 삼아 0차 적분 → 사다리꼴 (이 노트북) → Simpson 의 정확도 향상을 비교한다. 마지막에는 부정적분이 닫힌 형식으로 표현되지 않는 사례 (Bessel 함수 $I_0$, `50_exp_cos`)도 다룬다.<br>
Throughout this chapter we use $\int_0^1 e^x \, dx$ as the running example to compare 0th-order → Trapezoidal (this notebook) → Simpson. The chapter closes with an integral whose antiderivative has no closed form (Bessel $I_0$, in `50_exp_cos`).


부정적분이 알려져 있어 엄밀해를 비교 기준으로 쓸 수 있다.<br>
The antiderivative is known, so we have an exact reference value for comparison.

$$
\int_0^1 e^x \, dx = \left[ e^x \right]_0^1 = e - 1 \approx 1.7183
$$


In [ ]:
x_curve = np.linspace(0, 1, 100)
y_curve = np.exp(x_curve)

plt.fill_between(x_curve, y_curve, alpha=0.3)
plt.plot(x_curve, y_curve, label=r'$f(x) = e^x$')
plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc=0)
plt.grid(True)


## 사다리꼴 규칙<br>Trapezoid Rule



다음과 같은 사다리꼴을 생각해 보자.<br>Let's think about a trapezoid as follows.



In [ ]:
x_array = (0, 1)
y_array = (1, 2)

plt.fill_between(x_array, y_array)
plt.axis('equal')
plt.axis('off')

plt.text(-0.25, 0.5, '$y_i$')
plt.text(1.15, 1, '$y_{i+1}$')
plt.text(0.5, -0.3, '$\\Delta x$');



사다리꼴의 면적은 다음과 같다.<br>
Area of a trapezoid is as follows.



$$
a_i=\frac{1}{2} \left( y_i + y_{i+1} \right) \Delta x
$$



구간을 균등 분할 ($\Delta x = h$) 한 경우, 사다리꼴 적분과 0차 적분의 합을 직접 비교해 보자.<br>
With a uniform partition ($\Delta x = h$), let's compare the trapezoidal sum with the 0th-order sum directly.

$$
\begin{align}
\text{Trap} &= \sum_{k=0}^{n-1} \frac{1}{2}\left[f(x_k)+f(x_{k+1})\right] h
             = \frac{h}{2}\left[f(x_0)+f(x_n)\right] + h \sum_{k=1}^{n-1} f(x_k) \\
\text{Rect} &= \sum_{k=0}^{n-1} f(x_k)\,h
             = f(x_0)\,h + h \sum_{k=1}^{n-1} f(x_k) \\
\therefore \text{Trap} - \text{Rect} &= \frac{h}{2}\left[f(x_n) - f(x_0)\right]
\end{align}
$$

이 항등식이 이 장의 핵심 통찰이다. 적분 구간 양 끝값이 같은 경우 ($f(x_0) = f(x_n)$, 즉 0차 적분 노트북의 "특이 사례") 사다리꼴 적분과 0차 적분이 같은 값을 낸다 — 끝점 가중치 $\frac{h}{2}$ 가 양쪽에서 상쇄되기 때문이다.<br>
This identity is the central insight of this chapter. When the integrand takes the same value at both endpoints ($f(x_0) = f(x_n)$, i.e. the "special cases" of the 0th-order notebook), the trapezoidal and 0th-order sums collapse to identical numbers — the $\frac{h}{2}$ endpoint weights cancel symmetrically.

반대로 $e^x$ 는 $f(0) = 1 \ne e = f(1)$ 이므로 사다리꼴 적분이 0차 적분보다 실제로 정확하다.<br>
By contrast, $e^x$ has $f(0) = 1 \ne e = f(1)$, so trapezoidal genuinely improves on 0th-order.


## 1차 적분<br>First order numerical integration



마찬가지로 일정 간격으로 $x$ 좌표를 나누어 $f(x) = e^x$ 에 적용해 보자.<br>
Same as before, let's divide $x$ coordinates in a constant interval and apply the trapezoid rule to $f(x) = e^x$.


In [ ]:
n = 10
xi, xe = 0.0, 1.0

x_array_bar = np.linspace(xi, xe, n+1)
y_array_bar = np.exp(x_array_bar)
delta_x = x_array_bar[1] - x_array_bar[0]

x_curve = np.linspace(xi, xe, 200)
plt.fill_between(x_curve, np.exp(x_curve), alpha=0.3, label='exact')

for k in range(n):
    plt.fill(
        (x_array_bar[k], x_array_bar[k], x_array_bar[k+1], x_array_bar[k+1]),
        (0, y_array_bar[k], y_array_bar[k+1], 0),
        alpha=0.5, edgecolor='k',
    )

plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc=0)
plt.grid(True)


사다리꼴의 면적을 하나씩 구해서 더해보자.<br>Let's accumulate the area of trapezoids.



$$
    Area = \sum_{k=0}^{n-1} F_k
$$



$$
    F_k = \frac{\Delta x}{2}\left[f(x_k)+f(x_{k+1})\right]
$$



$$
    Area = \sum_{k=0}^{n-1}  \frac{1}{2}\left[f(x_k)+f(x_{k+1})\right] \Delta x
$$



In [ ]:
def get_delta_x(xi, xe, n):
    return (xe - xi) / n



In [ ]:
def num_int_1(f, xi, xe, n, b_verbose=False):
    # x coordinates of each interval
    x_array = np.linspace(xi, xe, n+1)

    # initialize result
    integration_result = 0.0

    x_k = x_array[0]
    # first height
    y_k = f(x_k)

    for k, x_k_plus_1 in enumerate(x_array[1:]):

        # second height of k-th step
        y_k_plus_1 = f(x_k_plus_1)

        # area of k-th trapezoid
        F_k = 0.5 * (y_k + y_k_plus_1) * (x_k_plus_1 - x_k)

        if b_verbose: print('k = %2d, F_k = %g' % (k, F_k))

        # accumulation
        integration_result += F_k

        # first height of next step
        x_k, y_k = x_k_plus_1, y_k_plus_1

    return integration_result



In [ ]:
n = 10
result = num_int_1(np.exp, 0.0, 1.0, n, b_verbose=True)
analytic = np.e - 1.0
print('result   =', result)
print('analytic =', analytic)
print('error    =', abs(result - analytic))


엄밀해 $e - 1 \approx 1.7183$ 에 더 가까운 값을 얻기 위해 더 잘게 나누어 보자.<br>
To obtain the result closer to the exact value $e - 1 \approx 1.7183$, let's divide with a narrower interval.


In [ ]:
n = 100
result = num_int_1(np.exp, 0.0, 1.0, n)
analytic = np.e - 1.0
print('result   =', result)
print('analytic =', analytic)
print('error    =', abs(result - analytic))


In [ ]:
%timeit -n 100 result = num_int_1(np.exp, 0.0, 1.0, n)


### 동적 탐색<br>Interactive Exploration

분할 수 $n$ 을 바꾸어 가며 사다리꼴이 $f(x) = e^x$ 에 어떻게 근사하는지, 오차가 어떻게 줄어드는지 관찰해 보자.<br>
Change the number of trapezoid panels $n$ and observe how the trapezoids approximate $f(x) = e^x$ — and how the error shrinks.


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_num_int_1_exp(n):
    # 자체 포함 / self-contained
    f = np.exp
    xi_loc, xe_loc = 0.0, 1.0
    analytic = np.e - 1.0

    x_curve = np.linspace(xi_loc, xe_loc, 200)
    y_curve = f(x_curve)

    x_bar = np.linspace(xi_loc, xe_loc, n + 1)
    y_bar = f(x_bar)

    plt.fill_between(x_curve, y_curve, alpha=0.3)

    for k in range(n):
        plt.fill(
            (x_bar[k], x_bar[k], x_bar[k+1], x_bar[k+1]),
            (0, y_bar[k], y_bar[k+1], 0),
            alpha=0.5, edgecolor='k',
        )

    area = num_int_1(f, xi_loc, xe_loc, n)
    plt.title(f'n = {n},  Area = {area:.6f},  Error = {abs(area - analytic):.2e}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.grid(True)
    plt.show()


if _ci:
    plot_num_int_1_exp(10)
else:
    interact(plot_num_int_1_exp, n=IntSlider(min=2, max=200, step=1, value=10,
                                              description='n :'));


사다리꼴 규칙의 절단 오차는 $-\frac{(b-a)}{12}\,h^2\,f''(\xi)$ 이다. $n$ 을 두 배로 하면 $h$ 가 절반이 되므로 오차는 약 $1/4$ 로 줄어든다.<br>
The trapezoid rule has truncation error $-\frac{(b-a)}{12}\,h^2\,f''(\xi)$. Doubling $n$ halves $h$, so the error shrinks by roughly $1/4$.



## 특이 사례<br>Special cases


위에서 유도한 항등식 $\text{Trap} - \text{Rect} = \frac{h}{2}(y_n - y_0)$ 의 결과를 다음 사례들에서 직접 확인할 수 있다. 모두 적분 구간 양 끝값이 같으므로 ($f(x_0) = f(x_n)$), 사다리꼴 적분과 0차 적분이 동일한 값을 내야 한다.<br>
The cases below let us verify the identity $\text{Trap} - \text{Rect} = \frac{h}{2}(y_n - y_0)$ derived above. All have $f(x_0) = f(x_n)$, so the trapezoidal and 0th-order sums must coincide.


### 반원<br>Half circle


다시 면적이 1인 반원을 생각해 보자.<br>Let's revisit the half circle with area 1.


$$
\begin{align}
    \pi r^2 &= 2 \\
    r^2 &= \frac{2}{\pi} \\
    r &= \sqrt{\frac{2}{\pi}}
\end{align}
$$



In [ ]:
r = np.sqrt(2.0 / np.pi)



In [ ]:
def half_circle(x):
    return np.sqrt(r**2 - x**2)



$$
y = \sqrt{r^2 - x^2}
$$



In [ ]:
import plot_num_int as pi



In [ ]:
pi.plot_a_half_circle_of_area(1)
pi.axis_equal_grid_True()



$n = 10$ 으로 분할한 사다리꼴 적분을 시각화해 보자.<br>Let's visualize the trapezoid rule with $n = 10$ panels.


In [ ]:
n = 10

pi.plot_half_circle_with_stems(n, 1)

# 사다리꼴의 좌표를 나눔 Find coordinates for the trapezoids
x_array_bar = np.linspace(-r, r, n+1)
y_array_bar = half_circle(x_array_bar)

# 각 사다리꼴의 폭 Width of each trapezoid
delta_x = x_array_bar[1] - x_array_bar[0]

# 일련의 사다리꼴을 그림 Plot a series of the trapezoids
xp, yp = x_array_bar[0], y_array_bar[0]
for x, y in zip(x_array_bar[1:], y_array_bar[1:]):
    plt.fill_between((xp, x), (yp, y), alpha=0.5, color=np.random.random((1, 3)))
    xp, yp = x, y

plt.axis('equal')
plt.grid(True)



In [ ]:
n = 10
result_half = num_int_1(half_circle, -r, r, n, b_verbose=True)
print('result =', result_half)


예상한 값 1에 더 비슷한 값을 얻기 위해 더 잘게 나누어 보자.<br>
To obtain the result closer to the expected value of 1, let's divide with a narrower interval.


In [ ]:
n = 100
result_half = num_int_1(half_circle, -r, r, n)
print('result =', result_half)


In [ ]:
%timeit -n 100 result_half = num_int_1(half_circle, -r, r, n)


#### 동적 탐색<br>Interactive Exploration


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_num_int_1(n):
    # 단위 면적 반원 / unit-area half-circle: r = sqrt(2/pi)
    r = np.sqrt(2.0 / np.pi)
    def half_circle(x):
        return np.sqrt(np.abs(r**2 - x**2))

    # smooth integrand curve / 부드러운 피적분함수 곡선
    x_curve = np.linspace(-r, r, 256)
    y_curve = half_circle(x_curve)

    # trapezoid panel vertices / 사다리꼴 분할점
    x_bar = np.linspace(-r, r, n + 1)
    y_bar = half_circle(x_bar)

    # underlying half-circle / 반원
    plt.fill_between(x_curve, y_curve, alpha=0.3)

    # trapezoid panels / 사다리꼴 채우기
    for k in range(n):
        plt.fill(
            (x_bar[k], x_bar[k], x_bar[k+1], x_bar[k+1]),
            (0, y_bar[k], y_bar[k+1], 0),
            alpha=0.5, edgecolor='k',
        )

    area = num_int_1(half_circle, -r, r, n)
    plt.title(f'n = {n},  Area = {area:.6f},  Error = {abs(area - 1):.2e}')
    plt.axis('equal')
    plt.grid(True)
    plt.show()


if _ci:
    # 위젯 대신 첫 단계만 렌더 / Render only the first step instead of using widget
    plot_num_int_1(10)
else:
    interact(
        plot_num_int_1,
        n=IntSlider(min=1, max=200, step=1, value=10, description='n :'),
    );


### $cos \theta$의 반 주기<br>Half period of $cos \theta$



In [ ]:
theta_deg = np.arange(180+1)
theta_rad = np.deg2rad(theta_deg)
s = np.sin(theta_rad)
c = np.cos(theta_rad)

plt.fill_between(theta_deg, c, color="C1", label="cos", alpha=0.5)

x_array_bar = np.linspace(0, 180, 10+1)
y_array_bar = np.cos(np.deg2rad(x_array_bar))

# 일련의 사다리꼴을 그림 Plot a series of the trapezoids
xp, yp = x_array_bar[0], y_array_bar[0]

for x, y in zip(x_array_bar[1:], y_array_bar[1:]):
    plt.fill_between((xp, x), (yp, y), alpha=0.5, color=np.random.random((1, 3)))
    xp, yp = x, y

plt.xticks(x_array_bar)

plt.xlabel(r"$\theta(deg)$")

plt.grid(True)



In [ ]:
n = 10
result_cos = num_int_1(np.cos, 0, np.pi, n, b_verbose=True)
print('result =', result_cos)



In [ ]:
theta_deg = np.arange(180+1)
theta_rad = np.deg2rad(theta_deg)
s = np.sin(theta_rad)
c = np.cos(theta_rad)

plt.fill_between(theta_deg, c, color="C1", label="cos", alpha=0.5)

x_array_bar = np.linspace(0, 180, 100+1)
y_array_bar = np.cos(np.deg2rad(x_array_bar))

# 일련의 사다리꼴을 그림 Plot a series of the trapezoids
xp, yp = x_array_bar[0], y_array_bar[0]

for x, y in zip(x_array_bar[1:], y_array_bar[1:]):
    plt.fill_between((xp, x), (yp, y), alpha=0.5, color=np.random.random((1, 3)))
    xp, yp = x, y

plt.xticks(x_array_bar[::10])

plt.xlabel(r"$\theta(deg)$")

plt.grid(True)



In [ ]:
n = 100
result_cos = num_int_1(np.cos, 0, np.pi, n)
print('result =', result_cos)



#### 동적 탐색<br>Interactive Exploration


분할 수 $n$ 을 바꾸어 가며 사다리꼴이 cos 곡선에 어떻게 근사하는지 직접 보자. cos 가 $\theta = \pi/2$ 에 대해 반대칭 (antisymmetric) 이므로 정확한 적분값은 $\int_0^\pi \cos\theta\,d\theta = 0$ 이고, 사다리꼴 규칙도 대칭성을 보존하므로 $n$ 값에 거의 무관하게 결과가 0 에 가깝다 &mdash; 시각적 근사 자체에 집중해 보자.<br>
Sweep $n$ and watch the trapezoid panels hug the cosine curve. The exact value $\int_0^\pi \cos\theta\,d\theta = 0$ comes from the antisymmetry of $\cos$ about $\theta = \pi/2$, and the trapezoid rule respects this symmetry &mdash; so the numeric result stays near zero regardless of $n$. Focus on the *visual* approximation here.


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_num_int_1_cos(n):
    # 부드러운 cos 곡선 / smooth cosine curve
    theta_deg_curve = np.linspace(0, 180, 256)
    y_curve = np.cos(np.deg2rad(theta_deg_curve))

    # 사다리꼴 분할점 / trapezoid panel vertices
    x_bar_deg = np.linspace(0, 180, n + 1)
    y_bar = np.cos(np.deg2rad(x_bar_deg))

    plt.fill_between(theta_deg_curve, y_curve, color="C1", alpha=0.3)

    # 사다리꼴 채우기 / trapezoid panels
    for k in range(n):
        plt.fill(
            (x_bar_deg[k], x_bar_deg[k], x_bar_deg[k+1], x_bar_deg[k+1]),
            (0, y_bar[k], y_bar[k+1], 0),
            alpha=0.5, edgecolor='k',
        )

    area = num_int_1(np.cos, 0, np.pi, n)
    plt.title(f'n = {n},  Area = {area:+.6f}  (exact = 0)')
    plt.xlabel(r"$\theta(deg)$")
    plt.grid(True)
    plt.show()


if _ci:
    # 위젯 대신 첫 단계만 렌더 / Render only the first step instead of using widget
    plot_num_int_1_cos(10)
else:
    interact(
        plot_num_int_1_cos,
        n=IntSlider(min=1, max=200, step=1, value=10, description='n :'),
    );


### 1/4 원<br>A quarter circle



In [ ]:
n = 10
result_quarter = num_int_1(half_circle, -r, 0, n, b_verbose=True)
print('result =', result_quarter)



In [ ]:
n = 10
result_quarter = num_int_1(half_circle, 0, r, n, b_verbose=True)
print('result =', result_quarter)



Let's compare with 0th order integration.<br>
0차 적분 결과와 비교해 보자.



In [ ]:
n = 100
result_quarter = num_int_1(half_circle, -r, 0, n)
print('result =', result_quarter)



## Exercises<br>연습 문제



Try this 1: Using the 1st order numerical integration, calculate the following. Please compare with the exact solution and the 0th order result<br>도전 과제 1: 1차 적분을 이용하여 다음을 계산하시오. 이론값, 0차 적분 값과 비교하시오

$$
\int_0^{2\pi}cos \theta d\theta
$$



Try this 2 : Compare the errors of the zeroth and first order integrations of the half circle example above using the same conditions. Duplicate the python function if necessary.<br>도전 과제 2 : 위 반원의 사례에 관해 다른 조건이 같을 때 0차 적분과 사다리꼴 적분의 오차를 비교해 보시오. 필요하면 해당 파이썬 함수를 복사하시오.



Try this 3 : Plot diagrams of shear force and bending moment of a cantilever with length $L=3m$ under distributed load $\omega=50sin\left(\frac{1}{2L}\pi x\right)[N/m]$.<br>도전 과제 3 : 길이 $L=3[m]$ 인 외팔보가 분포 하중 $\omega=50sin\left(\frac{1}{2L}\pi x\right)[N/m]$을 받고 있을 때 전단력과 굽힘모멘트 선도를 구하시오.



(ref : C 4.4, Pytel, Kiusalaas & Sharma, Mechanics of Materials, 2nd Ed, SI, Cengage Learning, 2011.)



## 함수형 프로그래밍<br>Functional programming



간격이 일정하다면 면적의 근사값을 다음과 같이 바꾸어 쓸 수 있다.<br>
If the interval $\Delta x$ is constant, we may rewrite the approximation of the area as follows.



$$
\begin{align}
    Area &= \sum_{k=0}^{n-1}  \frac{1}{2}\left[f(x_k)+f(x_{k+1})\right] \Delta x \\
   &= \Delta x \sum_{k=0}^{n-1}  \frac{1}{2}\left[f(x_k)+f(x_{k+1})\right]
\end{align}
$$



$$
\begin{align}
   \sum_{k=0}^{n-1}  \frac{1}{2}\left[f(x_k)+f(x_{k+1})\right] &= \frac{1}{2}\left[f(x_0)+f(x_1)\right] \\
   &+ \frac{1}{2}\left[f(x_1)+f(x_2)\right] \\
   &+ \frac{1}{2}\left[f(x_2)+f(x_3)\right] \\
   & \ldots \\
   &+ \frac{1}{2}\left[f(x_{n-2})+f(x_{n-1})\right] \\
   &+ \frac{1}{2}\left[f(x_{n-1})+f(x_{n})\right] \\
   &= \frac{1}{2}f(x_0) + \sum_{k=1}^{n-1}  f(x_k) + \frac{1}{2}f(x_{n}) \\
   &= \frac{1}{2}\left[f(x_0) + f(x_{n})\right] + \sum_{k=1}^{n-1}  f(x_k)
\end{align}
$$



$$
\begin{align}
    Area &= \Delta x \sum_{k=0}^{n-1}  \frac{1}{2}\left[f(x_k)+f(x_{k+1})\right] \\
    &= \Delta x \left[\frac{1}{2}\left[f(x_0) + f(x_{n})\right] + \sum_{k=1}^{n-1}  f(x_k)\right]
\end{align}
$$



할당문 없이 `sum()` 과 `map()` 함수로 구현해 보자.<br>
Instead of assignments, let's implement using `sum()` and `map()` functions.



In [ ]:
def num_int_1_functional(f, xi, xe, n):
    # get_delta_x() 함수 호출 횟수를 줄이기 위해 함수 안의 함수를 사용
    # To reduce the number of calling function get_delta_x(), define inner functions
    def with_delta_x(f, xi, n, delta_x=get_delta_x(xi, xe, n)):

        return delta_x * (
            0.5 * (f(xi) + f(xe))
            + sum(
                map(
                    f,
                    np.arange(xi + delta_x, xe - delta_x*0.1, delta_x),
                )
            )
        )

    return with_delta_x(f, xi, n)



In [ ]:
n = 100
result_func = num_int_1_functional(np.exp, 0.0, 1.0, n)
print('result_func =', result_func)


In [ ]:
import math
assert math.isclose(result, result_func), f"result = {result}, result_func = {result_func}"


In [ ]:
%timeit -n 100 result_func = num_int_1_functional(np.exp, 0.0, 1.0, n)


특이 사례 회귀 검사 / Smoke regression on the special case (half circle):


In [ ]:
# half_circle, r 은 위 '### 반원' 절에서 정의됨 / defined in the '### 반원' subsection above
result_func_smoke = num_int_1_functional(half_circle, -r, r, 100)
import math
assert math.isclose(result_func_smoke, 1.0, abs_tol=1e-2), result_func_smoke


## NumPy 벡터화<br>Vectorization of NumPy



In [ ]:
import matplotlib.pyplot as plt
import numpy as np



In [ ]:
def num_int_1_vector_with_delta_x(f, xi, xe, n, delta_x):
    return delta_x * (
        f(np.arange(xi+delta_x, xe-delta_x*0.5, get_delta_x(xi, xe, n))).sum()
        + 0.5 * f(np.array((xi, xe))).sum()
    )


def num_int_1_vector(f, xi, xe, n):

    return num_int_1_vector_with_delta_x(f, xi, xe, n, get_delta_x(xi, xe, n))



In [ ]:
n = 100
result_vect = num_int_1_vector(np.exp, 0.0, 1.0, n)
print('result_vect =', result_vect)


In [ ]:
assert 1e-7 > abs(result - result_vect), f"result = {result}, result_vect = {result_vect}"


In [ ]:
%timeit -n 100 result_vect = num_int_1_vector(np.exp, 0.0, 1.0, n)


특이 사례 회귀 검사 / Smoke regression on the special case (half circle):


In [ ]:
result_vect_smoke = num_int_1_vector(half_circle, -r, r, 100)
assert 1e-2 > abs(result_vect_smoke - 1.0), result_vect_smoke


## 시험<br>Test



아래는 함수가 맞게 작동하는지 확인함<br>
Following cells verify whether the functions work correctly.



In [ ]:
import math

# 주요 사례 / Primary case: e^x over [0, 1]
analytic = math.e - 1.0
for impl in (num_int_1, num_int_1_functional, num_int_1_vector):
    result_e = impl(np.exp, 0.0, 1.0, 100)
    assert math.isclose(result_e, analytic, abs_tol=1e-3), (impl.__name__, result_e)


In [ ]:
# 특이 사례 회귀 검사 / Smoke regression on the special case (half circle, area = 1)
r_test = np.sqrt(1.0 / np.pi)

def half_circle_test(x):
    return np.sqrt(np.maximum(r_test**2 - x**2, 0.0))

n = 10
for impl in (num_int_1, num_int_1_functional, num_int_1_vector):
    left  = 4.0 * impl(half_circle_test, -r_test, 0,      n)
    right = 4.0 * impl(half_circle_test,  0,      r_test, n)
    assert math.isclose(left,  1.0, abs_tol=0.02), (impl.__name__, 'left',  left)
    assert math.isclose(right, 1.0, abs_tol=0.02), (impl.__name__, 'right', right)


## Final Bell<br>마지막 종



In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");

